# Customer Value & Campaign ROI Analysis
**Dunnhumby — The Complete Journey (real US grocery loyalty card data)**

Sections:
1. Load & Clean Data
2. Customer Feature Engineering
3. RFM Segmentation
4. Campaign A/B Test & ROI (pre/post within-household uplift)
5. Customer Lifetime Value Prediction (XGBoost, Y1 features → Y2 spend)
6. Churn Prediction (Logistic Regression, AUC on holdout)
7. Cohort Retention Analysis
8. Financial Targeting Model
9. Visualisations
10. Save to Database

---

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

PLOT_DIR     = '.'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Assumptions (DMA 2023 industry benchmarks)
MAILER_COST    = {'TypeA': 1.50, 'TypeB': 0.80, 'TypeC': 0.50}
GROCERY_MARGIN = 0.27   # US grocery avg gross margin (FMI 2023)

# Time boundaries
MAX_DAY      = 711
CAL_END_DAY  = 365
CHURN_WINDOW = 90
CHURN_CUTOFF = MAX_DAY - CHURN_WINDOW

REF_DATE = pd.Timestamp('2013-01-01')
CAL_END  = REF_DATE + pd.to_timedelta(CAL_END_DAY - 1, unit='D')
OBS_END  = REF_DATE + pd.to_timedelta(MAX_DAY - 1,    unit='D')

RFM_ORDER  = ['Champions', 'Loyal', 'At Risk', 'Lost']
RFM_COLORS = {'Champions':'#2ecc71','Loyal':'#3498db','At Risk':'#f39c12','Lost':'#e74c3c'}
PALETTE    = ['#2c3e50','#8e44ad','#2980b9','#27ae60','#e74c3c']

print('Setup complete.')

---
## 1. Load & Clean Data

Seven CSV files from the Dunnhumby Complete Journey dataset:
- **transaction_data.csv** — household-level transactions
- **campaign_table.csv** — which households received which campaigns
- **campaign_desc.csv** — campaign metadata (type, start/end day)
- **coupon.csv** — coupon definitions
- **coupon_redempt.csv** — coupon redemption events
- **hh_demographic.csv** — household demographics
- **product.csv** — product metadata

In [ ]:
txn       = pd.read_csv('transaction_data.csv')
camp_tbl  = pd.read_csv('campaign_table.csv')
camp_desc = pd.read_csv('campaign_desc.csv')
coupon    = pd.read_csv('coupon.csv')
coupon_r  = pd.read_csv('coupon_redempt.csv')
demo      = pd.read_csv('hh_demographic.csv')
product   = pd.read_csv('product.csv')

for df in [txn, camp_tbl, camp_desc, coupon, coupon_r, demo, product]:
    df.columns = df.columns.str.lower()

txn['date'] = REF_DATE + pd.to_timedelta(txn['day'] - 1, unit='D')

# Basket-level aggregation
basket = (
    txn.groupby(['household_key','basket_id','day','week_no','date'])
    .agg(spend=('sales_value','sum'), n_items=('quantity','sum'))
    .reset_index()
)
basket_y1 = basket[basket['day'] <= CAL_END_DAY].copy()
basket_y2 = basket[basket['day'] >  CAL_END_DAY].copy()

print(f"Transactions  : {len(txn):,}")
print(f"Baskets       : {len(basket):,}")
print(f"Households    : {basket['household_key'].nunique():,}")
print(f"Date range    : {basket['date'].min().date()} -> {basket['date'].max().date()}")
print(f"Total revenue : ${txn['sales_value'].sum():,.2f}")
print(f"Avg basket    : ${basket['spend'].mean():.2f}")
print(f"Year 1 baskets: {len(basket_y1):,}  |  Year 2: {len(basket_y2):,}")

---
## 2. Customer Feature Engineering

Build one row per household with RFM signals, engagement metrics, income/age encoding, and year-split spend for CLV validation.

In [ ]:
spend_stats = basket.groupby('household_key').agg(
    total_spend   = ('spend',      'sum'),
    n_baskets     = ('basket_id',  'count'),
    avg_basket    = ('spend',      'mean'),
    first_day     = ('day',        'min'),
    last_day      = ('day',        'max'),
).reset_index()
spend_stats['tenure_days']  = spend_stats['last_day'] - spend_stats['first_day']
spend_stats['recency_days'] = MAX_DAY - spend_stats['last_day']
spend_stats['weekly_freq']  = spend_stats['n_baskets'] / (MAX_DAY / 7)

y1_spend = basket_y1.groupby('household_key')['spend'].sum().rename('y1_spend')
y2_spend = basket_y2.groupby('household_key')['spend'].sum().rename('y2_spend')

camp_engaged    = camp_tbl.groupby('household_key')['campaign'].count().rename('n_campaigns')
coupon_redeemed = coupon_r.groupby('household_key')['day'].count().rename('n_redemptions')

prod_dept = txn.merge(product[['product_id','department']], on='product_id', how='left')
dept_div  = prod_dept.groupby('household_key')['department'].nunique().rename('n_departments')

income_map = {
    'Under 15K':12.5, '15-24K':19.5, '25-34K':29.5, '35-49K':42,
    '50-74K':62, '75-99K':87, '100-124K':112, '125-149K':137,
    '150-174K':162, '175-199K':187, '200-249K':224, '250K+':275,
}
age_map = {'19-24':21.5,'25-34':29.5,'35-44':39.5,'45-54':49.5,'55-64':59.5,'65+':70}
demo['income_val'] = demo['income_desc'].map(income_map)
demo['age_val']    = demo['age_desc'].map(age_map)

customers = (
    spend_stats
    .join(y1_spend, on='household_key')
    .join(y2_spend, on='household_key')
    .join(camp_engaged, on='household_key')
    .join(coupon_redeemed, on='household_key')
    .join(dept_div, on='household_key')
    .merge(demo[['household_key','income_val','age_val']], on='household_key', how='left')
)
customers[['y1_spend','y2_spend','n_campaigns','n_redemptions']] = \
    customers[['y1_spend','y2_spend','n_campaigns','n_redemptions']].fillna(0)
customers['income_val']       = customers['income_val'].fillna(customers['income_val'].median())
customers['age_val']          = customers['age_val'].fillna(customers['age_val'].median())
customers['campaign_engaged'] = (customers['n_campaigns'] > 0).astype(int)

INCOME_MEDIAN = customers['income_val'].median()

print(f"Customer features built for {len(customers):,} households")
print(f"  Avg total spend   : ${customers['total_spend'].mean():,.2f}")
print(f"  Avg baskets       : {customers['n_baskets'].mean():.1f}")
print(f"  In campaigns      : {customers['campaign_engaged'].sum():,} "
      f"({customers['campaign_engaged'].mean()*100:.1f}%)")
print(f"  Coupon redeemers  : {(customers['n_redemptions']>0).sum():,}")
display(customers.describe().round(2))

---
## 3. RFM Segmentation

Score each household on Recency, Frequency, and Monetary value (quintile ranks 1–5). Combine into an RFM score and classify into 4 segments.

In [ ]:
customers['r_score'] = pd.qcut(
    customers['recency_days'].rank(method='first'), q=5, labels=[5,4,3,2,1]
).astype(int)
customers['f_score'] = pd.qcut(
    customers['n_baskets'].rank(method='first'), q=5, labels=[1,2,3,4,5]
).astype(int)
customers['m_score'] = pd.qcut(
    customers['total_spend'].rank(method='first'), q=5, labels=[1,2,3,4,5]
).astype(int)
customers['rfm_score'] = customers['r_score'] + customers['f_score'] + customers['m_score']

def rfm_segment(s):
    if s >= 13: return 'Champions'
    if s >= 10: return 'Loyal'
    if s >= 7:  return 'At Risk'
    return 'Lost'

customers['rfm_segment'] = customers['rfm_score'].apply(rfm_segment)

rfm_counts  = customers['rfm_segment'].value_counts()
rfm_pct     = rfm_counts / len(customers) * 100
rfm_profile = (
    customers.groupby('rfm_segment')
    .agg(n=('household_key','count'),
         avg_spend=('total_spend','mean'),
         avg_baskets=('n_baskets','mean'),
         avg_basket_size=('avg_basket','mean'),
         avg_recency=('recency_days','mean'),
         avg_rfm=('rfm_score','mean'))
    .reindex(RFM_ORDER).round(2)
)

print("RFM Segment Distribution:")
for seg in RFM_ORDER:
    n   = rfm_counts.get(seg, 0)
    avg = rfm_profile.loc[seg, 'avg_spend'] if seg in rfm_profile.index else 0
    print(f"  {seg:<12}: {n:5,}  ({rfm_pct.get(seg,0):.1f}%)  avg spend ${avg:,.2f}")

print("\nRFM Segment Profiles:")
display(rfm_profile)

---
## 4. Campaign A/B Test & ROI

**Method:** within-household pre/post comparison — each household serves as its own control.  
**Pre-period:** 8 weeks before campaign start.  
**Significance:** one-sample t-test (is weekly uplift > 0?).  
**Mailer costs:** TypeA=\$1.50, TypeB=\$0.80, TypeC=\$0.50 (DMA 2023 benchmarks).

In [ ]:
PRE_WEEKS = 8
uplift_rows = []

for _, cd in camp_desc.iterrows():
    cid       = cd['campaign']
    ctype     = cd['description']
    start     = cd['start_day']
    end       = cd['end_day']
    dur_weeks = max((end - start) / 7, 1)
    pre_start = max(start - PRE_WEEKS * 7, 1)
    pre_end   = start - 1

    treatment_hh = camp_tbl[camp_tbl['campaign'] == cid]['household_key'].values
    for hh in treatment_hh:
        hh_b = basket[basket['household_key'] == hh]
        pre_spend    = hh_b[(hh_b['day'] >= pre_start) & (hh_b['day'] <= pre_end)]['spend'].sum()
        during_spend = hh_b[(hh_b['day'] >= start)     & (hh_b['day'] <= end)    ]['spend'].sum()
        if pre_spend == 0:
            continue
        pre_weekly    = pre_spend / PRE_WEEKS
        during_weekly = during_spend / dur_weeks
        uplift_weekly = during_weekly - pre_weekly
        uplift_rows.append({
            'campaign': cid, 'campaign_type': ctype, 'household_key': hh,
            'pre_weekly_spend': round(pre_weekly, 2),
            'during_weekly_spend': round(during_weekly, 2),
            'uplift_weekly': round(uplift_weekly, 2),
            'dur_weeks': dur_weeks,
            'total_uplift': round(uplift_weekly * dur_weeks, 2),
        })

uplift_df = pd.DataFrame(uplift_rows)

camp_roi_rows = []
for ctype, grp in uplift_df.groupby('campaign_type'):
    n_hh         = grp['household_key'].nunique()
    avg_uplift_wk = grp['uplift_weekly'].mean()
    avg_dur      = grp['dur_weeks'].mean()
    total_uplift = grp['total_uplift'].sum()
    mailer_cost  = MAILER_COST[ctype]
    total_cost   = n_hh * mailer_cost
    gross_profit = total_uplift * GROCERY_MARGIN
    net_roi      = gross_profit / total_cost if total_cost > 0 else 0
    rev_roi      = total_uplift / total_cost if total_cost > 0 else 0
    t_stat, p_val = stats.ttest_1samp(grp['uplift_weekly'].dropna(), 0)
    camp_roi_rows.append({
        'campaign_type': ctype, 'n_households': n_hh,
        'avg_pre_weekly_spend': round(grp['pre_weekly_spend'].mean(), 2),
        'avg_during_weekly_spend': round(grp['during_weekly_spend'].mean(), 2),
        'avg_uplift_weekly': round(avg_uplift_wk, 2),
        'avg_duration_weeks': round(avg_dur, 1),
        'total_incremental_rev': round(total_uplift, 2),
        'total_mailer_cost': round(total_cost, 2),
        'gross_profit': round(gross_profit, 2),
        'revenue_roi': round(rev_roi, 2),
        'net_roi': round(net_roi, 2),
        'p_value': round(p_val, 4),
        'significant': int(p_val < 0.05),
        'mailer_cost_per_hh': mailer_cost,
    })

camp_roi = pd.DataFrame(camp_roi_rows).sort_values('net_roi', ascending=False)

print("Campaign ROI by Type:")
display(camp_roi[['campaign_type','n_households','avg_uplift_weekly',
                  'total_incremental_rev','revenue_roi','net_roi','p_value','significant']])

# Coupon analysis
total_campaign_hh = camp_tbl['household_key'].nunique()
redeeming_hh      = coupon_r['household_key'].nunique()
redemption_rate   = redeeming_hh / total_campaign_hh * 100

coupon_days = set(zip(coupon_r['household_key'], coupon_r['day']))
basket['is_coupon_day'] = basket.apply(
    lambda r: (r['household_key'], r['day']) in coupon_days, axis=1
)
coupon_avg  = basket[basket['is_coupon_day']]['spend'].mean()
regular_avg = basket[~basket['is_coupon_day']]['spend'].mean()

print(f"\nCoupon Analysis:")
print(f"  Redemption rate            : {redemption_rate:.1f}% of campaign households")
print(f"  Avg basket on coupon day   : ${coupon_avg:.2f}")
print(f"  Avg basket on regular day  : ${regular_avg:.2f}")
print(f"  Basket uplift on coupon day: ${coupon_avg - regular_avg:+.2f}")

---
## 5. CLV Prediction — XGBoost

**Model:** XGBoost Regressor trained on 17 Year-1 behavioural + demographic features.  
**Target:** Year-2 actual spend (held-out holdout set).  
**Split:** 80/20 random. Early stopping on a 20% internal validation slice to prevent overfitting.  
**Baseline:** Naive predictor (Y2 = Y1 spend).

In [ ]:
y1_stats = basket_y1.groupby('household_key').agg(
    y1_total_spend = ('spend',     'sum'),
    y1_n_baskets   = ('basket_id', 'count'),
    y1_avg_basket  = ('spend',     'mean'),
    y1_std_basket  = ('spend',     'std'),
    y1_first_day   = ('day',       'min'),
    y1_last_day    = ('day',       'max'),
).reset_index()
y1_stats['y1_recency']    = CAL_END_DAY - y1_stats['y1_last_day']
y1_stats['y1_tenure']     = y1_stats['y1_last_day'] - y1_stats['y1_first_day']
y1_stats['y1_std_basket'] = y1_stats['y1_std_basket'].fillna(0)
y1_stats['y1_spend_cv']   = np.where(
    y1_stats['y1_avg_basket'] > 0,
    y1_stats['y1_std_basket'] / y1_stats['y1_avg_basket'], 0
)

HALF_DAY = CAL_END_DAY // 2
y1_h1 = basket_y1[basket_y1['day'] <= HALF_DAY].groupby('household_key')['spend'].sum().rename('y1_spend_h1')
y1_h2 = basket_y1[basket_y1['day'] >  HALF_DAY].groupby('household_key')['spend'].sum().rename('y1_spend_h2')
y1_stats = y1_stats.join(y1_h1, on='household_key').join(y1_h2, on='household_key')
y1_stats[['y1_spend_h1','y1_spend_h2']] = y1_stats[['y1_spend_h1','y1_spend_h2']].fillna(0)
y1_stats['y1_spend_trend'] = np.where(
    y1_stats['y1_spend_h1'] > 0, y1_stats['y1_spend_h2'] / y1_stats['y1_spend_h1'], 1.0
)

y1_last4wk = (
    basket_y1[basket_y1['day'] >= CAL_END_DAY - 28]
    .groupby('household_key')['spend'].sum().rename('y1_last4wk_spend')
)
y1_disc = (
    txn[txn['day'] <= CAL_END_DAY].groupby('household_key')['retail_disc']
    .sum().abs().rename('y1_total_disc')
)
y1_stats = y1_stats.join(y1_last4wk, on='household_key').join(y1_disc, on='household_key')
y1_stats['y1_last4wk_spend'] = y1_stats['y1_last4wk_spend'].fillna(0)
y1_stats['y1_total_disc']    = y1_stats['y1_total_disc'].fillna(0)
y1_stats['y1_disc_ratio']    = np.where(
    y1_stats['y1_total_spend'] > 0,
    y1_stats['y1_total_disc'] / y1_stats['y1_total_spend'], 0
)

y1_coupons = (
    coupon_r[coupon_r['day'] <= CAL_END_DAY]
    .groupby('household_key')['day'].count().rename('y1_redemptions')
)
y2_target = basket_y2.groupby('household_key')['spend'].sum().rename('y2_spend')

model_df = (
    customers[['household_key','n_departments','campaign_engaged','income_val','age_val']]
    .merge(y1_stats, on='household_key', how='left')
    .join(y1_coupons, on='household_key')
    .join(y2_target,  on='household_key')
)
model_df['y1_redemptions'] = model_df['y1_redemptions'].fillna(0)
model_df['y2_spend']       = model_df['y2_spend'].fillna(0)
model_df = model_df.set_index('household_key')

FEATURE_COLS = [
    'y1_total_spend', 'y1_n_baskets',  'y1_avg_basket',   'y1_std_basket',
    'y1_spend_cv',    'y1_recency',    'y1_tenure',       'y1_spend_h1',
    'y1_spend_h2',    'y1_spend_trend','y1_last4wk_spend','y1_disc_ratio',
    'n_departments',  'campaign_engaged','y1_redemptions', 'income_val', 'age_val',
]

X = model_df[FEATURE_COLS].fillna(0)
y = model_df['y2_spend']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val         = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

xgb = XGBRegressor(
    n_estimators=1000, learning_rate=0.03, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
    objective='reg:absoluteerror', n_jobs=-1, random_state=RANDOM_STATE,
    verbosity=0, early_stopping_rounds=50,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print(f"Best iteration: {xgb.best_iteration} trees")

y_pred_test = xgb.predict(X_test)
y_pred_all  = xgb.predict(X)

abs_err        = np.abs(y_pred_test - y_test.values)
mae            = abs_err.mean()
median_ae      = np.median(abs_err)
rmse           = np.sqrt((abs_err**2).mean())
r2             = r2_score(y_test, y_pred_test)
top_n          = max(1, int(len(y_test) * 0.2))
top_actual     = set(y_test.nlargest(top_n).index)
top_predicted  = set(pd.Series(y_pred_test, index=y_test.index).nlargest(top_n).index)
top20_precision = len(top_actual & top_predicted) / top_n

print(f"\nTest-Set Metrics  (train={len(X_train):,} | test={len(X_test):,})")
print(f"  MAE              : ${mae:.2f}")
print(f"  Median Abs Error : ${median_ae:.2f}")
print(f"  RMSE             : ${rmse:.2f}")
print(f"  R²               : {r2:.4f}")
print(f"  Top-20% Precision: {top20_precision*100:.1f}%")

feat_imp = pd.DataFrame({'feature': FEATURE_COLS, 'importance': xgb.feature_importances_})\
             .sort_values('importance', ascending=False).reset_index(drop=True)
print("\nTop-10 Feature Importances:")
display(feat_imp.head(10))

clv_results = pd.DataFrame({
    'household_key':   model_df.index,
    'predicted_clv':   y_pred_all,
    'actual_y2_spend': y.values,
    'in_test_set':     model_df.index.isin(X_test.index),
})
clv_results['abs_error'] = (clv_results['predicted_clv'] - clv_results['actual_y2_spend']).abs()

customers = customers.merge(clv_results[['household_key','predicted_clv']], on='household_key', how='left')
customers['predicted_clv'] = customers['predicted_clv'].fillna(clv_results['predicted_clv'].median())

naive_pred     = model_df.loc[X_test.index, 'y1_total_spend']
naive_abs_err  = np.abs(naive_pred.values - y_test.values)
naive_mae      = naive_abs_err.mean()
rf_improvement = (1 - mae / naive_mae) * 100
print(f"\nNaive MAE: ${naive_mae:.2f}  |  XGBoost MAE: ${mae:.2f}  |  Improvement: {rf_improvement:.1f}%")

clv_metrics = {
    'model': 'XGBoost + log-transform (n=500)',
    'median_ae': round(median_ae, 2), 'mae': round(mae, 2), 'rmse': round(rmse, 2),
    'r2': round(r2, 4), 'top20_precision': round(top20_precision, 4),
    'median_predicted_clv': round(float(np.median(y_pred_all)), 2),
    'mean_predicted_clv':   round(float(y_pred_all.mean()), 2),
    'n_train': len(X_train), 'n_test': len(X_test),
    'naive_mae': round(naive_mae, 2), 'naive_median_ae': round(np.median(naive_abs_err), 2),
    'rf_improvement_pct': round(rf_improvement, 1),
}

---
## 6. Churn Prediction — Logistic Regression

**Definition:** no purchase in the last 90 days (days 621–711).  
**Note:** `recency_days` is excluded from features because it directly encodes the label.

In [ ]:
customers['is_churned'] = (customers['last_day'] < CHURN_CUTOFF).astype(int)
churn_rate = customers['is_churned'].mean()
print(f"Overall churn rate: {churn_rate*100:.1f}%  (no purchase in last {CHURN_WINDOW} days)")

feature_cols = ['n_baskets','total_spend','avg_basket',
                'tenure_days','campaign_engaged','n_departments',
                'income_val','age_val','weekly_freq']
X = customers[feature_cols].fillna(0)
y = customers['is_churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
lr.fit(X_train_sc, y_train)

y_prob = lr.predict_proba(X_test_sc)[:, 1]
y_pred = lr.predict(X_test_sc)
auc    = roc_auc_score(y_test, y_prob)

print(f"\nTest set AUC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))

X_all_sc = scaler.transform(X.fillna(0))
customers['churn_probability'] = lr.predict_proba(X_all_sc)[:, 1]

high_risk       = customers[customers['churn_probability'] >= 0.70]
revenue_at_risk = high_risk['predicted_clv'].sum()
n_high_risk     = len(high_risk)

print(f"\nHigh-risk customers (prob >= 70%): {n_high_risk:,}")
print(f"Revenue at risk (predicted CLV)  : ${revenue_at_risk:,.2f}")

churn_coef = pd.DataFrame({'feature': feature_cols, 'coefficient': lr.coef_[0]})\
               .sort_values('coefficient', key=abs, ascending=False)
print("\nTop churn predictors:")
display(churn_coef.head(6))

---
## 7. Cohort Retention Analysis

Cohort = month of first purchase. Only cohorts with ≥ 20 customers are kept.

In [ ]:
customers['first_quarter'] = pd.to_datetime(
    REF_DATE + pd.to_timedelta(customers['first_day'] - 1, unit='D')
).dt.to_period('M').astype(str)

cohort = (
    customers.groupby('first_quarter')
    .agg(
        n_customers  = ('household_key',    'count'),
        avg_spend    = ('total_spend',      'mean'),
        avg_baskets  = ('n_baskets',        'mean'),
        avg_clv      = ('predicted_clv',    'mean'),
        churn_rate   = ('is_churned',       'mean'),
        pct_campaign = ('campaign_engaged', 'mean'),
    )
    .reset_index()
    .sort_values('first_quarter')
)
cohort = cohort[cohort['n_customers'] >= 20]

print("Cohort Analysis:")
display(cohort.round(2))

best_cohort  = cohort.loc[cohort['avg_spend'].idxmax(), 'first_quarter']
worst_cohort = cohort.loc[cohort['avg_spend'].idxmin(), 'first_quarter']
print(f"\nHighest-spend cohort : {best_cohort}  (${cohort['avg_spend'].max():,.2f} avg)")
print(f"Lowest-spend cohort  : {worst_cohort}  (${cohort['avg_spend'].min():,.2f} avg)")

---
## 8. Financial Targeting Model

Compare three mailing strategies using the campaign uplift data and CLV predictions:
- **Blanket TypeA** — send to every household
- **Targeted (RFM)** — TypeA to At-Risk, TypeB to Champions + Loyal, nothing to Lost
- **CLV-Gated** — only send where predicted CLV > 2× mailer cost

In [ ]:
seg_stats = customers.groupby('rfm_segment').agg(
    n          = ('household_key', 'count'),
    avg_clv    = ('predicted_clv', 'mean'),
    churn_rate = ('is_churned',    'mean'),
).reindex(RFM_ORDER)

typeA_row = camp_roi[camp_roi['campaign_type']=='TypeA'].iloc[0] if len(camp_roi[camp_roi['campaign_type']=='TypeA']) else None
typeB_row = camp_roi[camp_roi['campaign_type']=='TypeB'].iloc[0] if len(camp_roi[camp_roi['campaign_type']=='TypeB']) else None
typeA_rev_roi = typeA_row['revenue_roi'] if typeA_row is not None else 2.5
typeB_rev_roi = typeB_row['revenue_roi'] if typeB_row is not None else 1.8

n_champ  = int(rfm_counts.get('Champions', 0))
n_loyal  = int(rfm_counts.get('Loyal',     0))
n_atrisk = int(rfm_counts.get('At Risk',   0))
n_lost   = int(rfm_counts.get('Lost',      0))
n_total  = len(customers)

blanket_cost   = n_total * MAILER_COST['TypeA']
blanket_uplift = ((typeA_row['avg_uplift_weekly'] * typeA_row['avg_duration_weeks']
                   if typeA_row is not None else 0) * n_total)
blanket_roi    = blanket_uplift / blanket_cost if blanket_cost > 0 else 0

targeted_cost   = (n_atrisk * MAILER_COST['TypeA'] + (n_champ + n_loyal) * MAILER_COST['TypeB'])
targeted_uplift = (
    (typeA_row['avg_uplift_weekly'] * typeA_row['avg_duration_weeks'] if typeA_row is not None else 0) * n_atrisk +
    (typeB_row['avg_uplift_weekly'] * typeB_row['avg_duration_weeks'] if typeB_row is not None else 0) * (n_champ + n_loyal)
)
targeted_roi = targeted_uplift / targeted_cost if targeted_cost > 0 else 0

clv_gate     = customers[customers['predicted_clv'] > 2 * MAILER_COST['TypeA']].copy()
gated_n      = len(clv_gate)
gated_cost   = gated_n * MAILER_COST['TypeA']
gated_uplift = (typeA_row['avg_uplift_weekly'] * typeA_row['avg_duration_weeks']
                if typeA_row is not None else 0) * gated_n
gated_roi    = gated_uplift / gated_cost if gated_cost > 0 else 0

cost_saving  = blanket_cost - targeted_cost
rev_retained = targeted_uplift / blanket_uplift * 100 if blanket_uplift > 0 else 0

print(f"Assumptions: TypeA=${MAILER_COST['TypeA']}, TypeB=${MAILER_COST['TypeB']}, "
      f"TypeC=${MAILER_COST['TypeC']} (DMA 2023) | Grocery margin: {GROCERY_MARGIN*100:.0f}% (FMI 2023)")
print(f"\n{'Metric':<32} {'Blanket TypeA':>14} {'Targeted':>14} {'CLV-Gated':>14}")
print("-"*76)
print(f"{'Households targeted':<32} {n_total:>14,} {n_champ+n_loyal+n_atrisk:>14,} {gated_n:>14,}")
print(f"{'Total mailer cost ($)':<32} ${blanket_cost:>13,.0f} ${targeted_cost:>13,.0f} ${gated_cost:>13,.0f}")
print(f"{'Incremental revenue ($)':<32} ${blanket_uplift:>13,.0f} ${targeted_uplift:>13,.0f} ${gated_uplift:>13,.0f}")
print(f"{'Revenue ROI':<32} {blanket_roi:>14.1f}x {targeted_roi:>14.1f}x {gated_roi:>14.1f}x")
print(f"\nTargeted strategy retains {rev_retained:.0f}% of uplift at {cost_saving/blanket_cost*100:.0f}% lower cost.")

---
## 9. Visualisations

Eight charts covering data overview, RFM segments, campaign ROI, CLV predictions, churn, cohort, financial model, and category analysis. Charts are saved as PNGs and displayed inline.

In [ ]:
# Figure 1: Data Overview
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Data Overview — Dunnhumby Complete Journey', fontsize=14, fontweight='bold')

axes[0].hist(customers['total_spend'].clip(upper=10000), bins=30, color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[0].set_title('Total Spend Distribution ($)')
axes[0].set_xlabel('Total 2-Year Spend ($)')
axes[0].axvline(customers['total_spend'].median(), color='red', linestyle='--',
                label=f"Median ${customers['total_spend'].median():,.0f}")
axes[0].legend()

axes[1].hist(customers['n_baskets'].clip(upper=300), bins=30, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].set_title('Shopping Frequency (Baskets)')
axes[1].set_xlabel('Number of Baskets')
axes[1].axvline(customers['n_baskets'].median(), color='red', linestyle='--',
                label=f"Median {customers['n_baskets'].median():.0f}")
axes[1].legend()

axes[2].hist(basket['spend'].clip(upper=150), bins=30, color=PALETTE[3], edgecolor='white', alpha=0.85)
axes[2].set_title('Basket Size Distribution ($)')
axes[2].set_xlabel('Basket Spend ($)')
axes[2].axvline(basket['spend'].median(), color='red', linestyle='--',
                label=f"Median ${basket['spend'].median():.2f}")
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig1_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: RFM Segments
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RFM Customer Segmentation', fontsize=13, fontweight='bold')

sizes  = [rfm_counts.get(s, 0) for s in RFM_ORDER]
colors = [RFM_COLORS[s] for s in RFM_ORDER]
axes[0].pie(sizes, labels=[f'{s}\n{rfm_counts.get(s,0):,}' for s in RFM_ORDER],
            autopct='%1.0f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Segment Sizes')

avg_spends = [rfm_profile.loc[s,'avg_spend'] for s in RFM_ORDER if s in rfm_profile.index]
bars = axes[1].bar(RFM_ORDER, avg_spends, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Avg 2-Year Spend per Segment ($)')
axes[1].set_ylabel('Avg Spend ($)')
for bar, v in zip(bars, avg_spends):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'${v:,.0f}', ha='center', fontweight='bold', fontsize=10)

x, w = np.arange(len(RFM_ORDER)), 0.25
r_sc = [rfm_profile.loc[s,'avg_spend']/rfm_profile['avg_spend'].max()*5 for s in RFM_ORDER if s in rfm_profile.index]
f_sc = [rfm_profile.loc[s,'avg_baskets']/rfm_profile['avg_baskets'].max()*5 for s in RFM_ORDER if s in rfm_profile.index]
axes[2].bar(x-w, r_sc, width=w, label='Monetary (norm)', color='#2ecc71', alpha=0.85)
axes[2].bar(x,   f_sc, width=w, label='Frequency (norm)', color='#3498db', alpha=0.85)
axes[2].set_xticks(x); axes[2].set_xticklabels(RFM_ORDER)
axes[2].set_title('Normalised Spend & Frequency by Segment')
axes[2].set_ylabel('Score (0–5)'); axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig2_rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Campaign ROI
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Campaign ROI — Pre/Post Within-Household Uplift Analysis', fontsize=12, fontweight='bold')

camp_colors = {'TypeA':'#e74c3c','TypeB':'#f39c12','TypeC':'#3498db'}
clrs = [camp_colors.get(t,'#888') for t in camp_roi['campaign_type']]

bars = axes[0].bar(camp_roi['campaign_type'], camp_roi['avg_uplift_weekly'],
                   color=clrs, edgecolor='white', width=0.4)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Avg Weekly Spend Uplift per Household ($)')
axes[0].set_ylabel('Uplift ($/week)')
for bar, v, p in zip(bars, camp_roi['avg_uplift_weekly'], camp_roi['p_value']):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height() + (0.05 if v >= 0 else -0.2),
                 f'${v:+.2f}{sig}', ha='center', fontsize=11, fontweight='bold')

x, w = np.arange(len(camp_roi)), 0.3
axes[1].bar(x-w/2, camp_roi['total_mailer_cost'],    width=w, label='Mailer Cost ($)',        color='#e74c3c', alpha=0.85)
axes[1].bar(x+w/2, camp_roi['total_incremental_rev'], width=w, label='Incremental Revenue ($)', color='#27ae60', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(camp_roi['campaign_type'])
axes[1].set_title('Total Cost vs Incremental Revenue by Campaign Type')
axes[1].set_ylabel('Amount ($)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig3_campaign_roi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: CLV Predictions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'CLV Model — XGBoost  |  MAE=${mae:.2f}  R²={r2:.3f}  Top-20% Precision={top20_precision*100:.0f}%',
             fontsize=11, fontweight='bold')

sample = clv_results.sample(min(500, len(clv_results)), random_state=RANDOM_STATE)
axes[0].scatter(sample['predicted_clv'], sample['actual_y2_spend'], alpha=0.4, color=PALETTE[2], s=15)
lim = max(sample['predicted_clv'].max(), sample['actual_y2_spend'].max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfect prediction')
axes[0].set_xlabel('Predicted CLV'); axes[0].set_ylabel('Actual Year-2 Spend')
axes[0].set_title('Predicted vs Actual (sample n=500)'); axes[0].legend(fontsize=8)

axes[1].hist(clv_results['predicted_clv'].clip(upper=clv_results['predicted_clv'].quantile(0.99)),
             bins=30, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].set_title('Predicted CLV Distribution'); axes[1].set_xlabel('Predicted CLV ($)')
axes[1].axvline(clv_results['predicted_clv'].median(), color='red', linestyle='--',
                label=f"Median ${clv_results['predicted_clv'].median():.0f}")
axes[1].legend()

fi_top = feat_imp.head(8).reset_index(drop=True)
axes[2].barh(fi_top['feature'].values[::-1], fi_top['importance'].values[::-1],
             color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[2].set_title('Top Feature Importances'); axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig4_clv_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Churn Analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Churn Prediction — Logistic Regression  |  AUC={auc:.4f}  Churn Rate={churn_rate*100:.1f}%',
             fontsize=11, fontweight='bold')

axes[0].hist(customers['churn_probability'], bins=30, color=PALETTE[4], edgecolor='white', alpha=0.85)
axes[0].axvline(0.70, color='red', linestyle='--', label='High-risk threshold (0.70)')
axes[0].set_title('Churn Probability Distribution')
axes[0].set_xlabel('Churn Probability'); axes[0].set_ylabel('Number of Households'); axes[0].legend()

churn_by_seg = customers.groupby('rfm_segment')['is_churned'].mean().reindex(RFM_ORDER)
bars = axes[1].bar(RFM_ORDER, churn_by_seg.values * 100, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Churn Rate by RFM Segment (%)'); axes[1].set_ylabel('Churn Rate (%)')
for bar, v in zip(bars, churn_by_seg.values*100):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

feat_plot   = churn_coef.head(8).copy()
feat_colors = ['#e74c3c' if c > 0 else '#2980b9' for c in feat_plot['coefficient']]
axes[2].barh(feat_plot['feature'], feat_plot['coefficient'], color=feat_colors)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Churn Model Coefficients'); axes[2].set_xlabel('Logistic Regression Coefficient')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig5_churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Cohort Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cohort Retention & Spend by Join Month', fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(cohort['first_quarter'], cohort['avg_spend'], marker='o', color='#2980b9', linewidth=2, label='Avg 2yr Spend')
ax.set_xlabel('Join Month'); ax.set_ylabel('Avg 2-Year Spend ($)', color='#2980b9')
ax.tick_params(axis='x', rotation=45)
ax2 = ax.twinx()
ax2.plot(cohort['first_quarter'], cohort['churn_rate']*100,
         marker='s', color='#e74c3c', linewidth=2, linestyle='--', label='Churn Rate %')
ax2.set_ylabel('Churn Rate (%)', color='#e74c3c')
ax2.tick_params(axis='y', colors='#e74c3c')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=9)
ax.set_title('Avg Spend & Churn Rate by Join Cohort')

hm = cohort.set_index('first_quarter')[['avg_spend','avg_baskets','avg_clv','churn_rate']].T.astype(float)
sns.heatmap(hm, ax=axes[1], cmap='Blues', annot=True, fmt='.1f',
            linewidths=0.5, cbar_kws={'label':'Value'})
axes[1].set_title('Cohort Metrics Heatmap')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig6_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Financial Model
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Financial Targeting Scenarios — Real Uplift Numbers', fontsize=12, fontweight='bold')

scenarios = ['Blanket\nTypeA', 'Targeted\n(RFM)', 'CLV-Gated']
costs   = [blanket_cost,   targeted_cost,   gated_cost]
uplifts = [blanket_uplift, targeted_uplift, gated_uplift]
rois    = [blanket_roi,    targeted_roi,    gated_roi]
clrs_sc = ['#e74c3c','#f39c12','#27ae60']

bars = axes[0].bar(scenarios, rois, color=clrs_sc, edgecolor='white', width=0.5)
axes[0].axhline(1, color='black', linestyle='--', linewidth=0.8, label='Break-even (1x)')
axes[0].set_title('Revenue ROI by Strategy'); axes[0].set_ylabel('Revenue ROI (Uplift / Cost)')
axes[0].legend()
for bar, v in zip(bars, rois):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{v:.1f}x', ha='center', fontsize=13, fontweight='bold')

x, w = np.arange(3), 0.35
axes[1].bar(x-w/2, costs,   width=w, label='Mailer Cost ($)',     color='#e74c3c', alpha=0.85)
axes[1].bar(x+w/2, uplifts, width=w, label='Incremental Rev ($)', color='#27ae60', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(scenarios)
axes[1].set_title('Cost vs Incremental Revenue ($)')
axes[1].set_ylabel('Amount ($)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig7_financial_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8: Category Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Product Category Analysis', fontsize=12, fontweight='bold')

cat_spend = (
    txn.merge(product[['product_id','department']], on='product_id', how='left')
    .groupby('department')['sales_value'].sum()
    .sort_values(ascending=False).head(10)
)
bars = axes[0].barh(cat_spend.index[::-1], cat_spend.values[::-1], color=PALETTE[2], edgecolor='white')
axes[0].set_title('Top 10 Departments by Revenue'); axes[0].set_xlabel('Total Revenue ($)')
for bar, v in zip(bars, cat_spend.values[::-1]):
    axes[0].text(bar.get_width()+5000, bar.get_y()+bar.get_height()/2,
                 f'${v:,.0f}', va='center', fontsize=8)

nat_vs_priv = txn.merge(product[['product_id','brand']], on='product_id', how='left')
nat_vs_priv = nat_vs_priv.groupby('brand')['sales_value'].sum().sort_values(ascending=False)
axes[1].bar(nat_vs_priv.index, nat_vs_priv.values / 1e6,
            color=[PALETTE[2] if b=='National' else PALETTE[1] for b in nat_vs_priv.index],
            edgecolor='white')
axes[1].set_title('Revenue by Brand Type (National vs Private Label)')
axes[1].set_ylabel('Revenue ($M)'); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/fig8_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Executive Summary

| Finding | Result |
|---|---|
| **RFM segmentation** | 4 value tiers — Champions, Loyal, At Risk, Lost |
| **Campaign ROI** | Varies significantly by mailer type (pre/post uplift) |
| **CLV model** | XGBoost on 17 Y1 features, validated on Y2 holdout |
| **Churn model** | Logistic regression, AUC reported on 20% holdout |
| **Targeting strategy** | Targeted/CLV-Gated retains most revenue at lower cost |

---
## 10. Save to Database

In [ ]:
from database import (
    build_database, save_rfm, save_clv, save_churn,
    save_campaign_roi, save_cohort, save_financial, save_clv_metrics,
    save_feature_importance, get_connection,
)

build_database(force_rebuild=True)

save_rfm(customers)
save_clv(clv_results[['household_key','predicted_clv','actual_y2_spend','abs_error','in_test_set']])
save_feature_importance(feat_imp)

churn_save = customers[['household_key','churn_probability','is_churned',
                         'rfm_segment','predicted_clv']].copy()
save_churn(churn_save)
save_campaign_roi(camp_roi)
save_cohort(cohort)

financial_scenarios = [
    {'scenario':'Blanket TypeA',  'n_households':n_total,
     'mailer_cost':round(blanket_cost),   'incremental_revenue':round(blanket_uplift),
     'revenue_roi':round(blanket_roi,2),  'cost_saving':0},
    {'scenario':'Targeted (RFM)', 'n_households':n_champ+n_loyal+n_atrisk,
     'mailer_cost':round(targeted_cost),  'incremental_revenue':round(targeted_uplift),
     'revenue_roi':round(targeted_roi,2), 'cost_saving':round(cost_saving)},
    {'scenario':'CLV-Gated',      'n_households':gated_n,
     'mailer_cost':round(gated_cost),     'incremental_revenue':round(gated_uplift),
     'revenue_roi':round(gated_roi,2),    'cost_saving':round(blanket_cost-gated_cost)},
]
save_financial(financial_scenarios)
save_clv_metrics(clv_metrics)

con = get_connection()
uplift_df.to_sql('campaign_uplift_detail', con, if_exists='replace', index=False)
con.commit(); con.close()

print('All results saved to grocery_analysis.db')
print('Run: streamlit run dashboard.py')